In [0]:
df = spark.table("dbw_kroger_insights.silver.ab_test").toPandas()

print(df.shape)
print(df["test_group"].value_counts())
df.head()

In [0]:
from scipy import stats

observed = df["test_group"].value_counts().sort_index()   # control, treatment
expected = [observed.sum() / 2] * 2                       # designed 50/50 split

chi2, p_value = stats.chisquare(f_obs=observed.values, f_exp=expected)

print(observed)
print(f"Chi-square: {chi2:.4f}")
print(f"p-value:    {p_value:.4f}")
print("SRM detected – investigate before analysing!" if p_value < 0.01 else "No SRM – split is healthy.")

In [0]:
import numpy as np
from scipy import stats

# --- Step 3: conversion by group ---
summary = df.groupby("test_group")["converted"].agg(users="count", conversions="sum")
summary["conversion_rate"] = summary["conversions"] / summary["users"]
print(summary)

n_c, x_c = summary.loc["control",   ["users", "conversions"]]
n_t, x_t = summary.loc["treatment", ["users", "conversions"]]
p_c, p_t = x_c / n_c, x_t / n_t
diff = p_t - p_c                                   # treatment minus control

# --- Two-proportion z-test (H0: no difference) ---
p_pool  = (x_c + x_t) / (n_c + n_t)                # pooled rate, assuming H0 is true
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
z       = diff / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))         # two-sided

# --- Step 4: 95% confidence interval for the difference ---
se_diff  = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)   # unpooled SE for the CI
z_crit   = stats.norm.ppf(0.975)                         # 1.96
ci_low, ci_high = diff - z_crit*se_diff, diff + z_crit*se_diff

print(f"\nControl conversion:   {p_c:.4%}")
print(f"Treatment conversion: {p_t:.4%}")
print(f"Difference:           {diff*100:+.3f} pp  ({diff/p_c:+.2%} relative)")
print(f"z = {z:.3f},  p-value = {p_value:.4f}")
print(f"95% CI for difference: [{ci_low*100:+.3f} pp, {ci_high*100:+.3f} pp]")

In [0]:
# --- Step 5: Power / Minimum Detectable Effect ---
alpha, power = 0.05, 0.80
z_alpha = stats.norm.ppf(1 - alpha/2)     # 1.96
z_beta  = stats.norm.ppf(power)           # 0.84

n_per_group = min(n_c, n_t)
p_base = p_c

mde = (z_alpha + z_beta) * np.sqrt(2 * p_base * (1 - p_base) / n_per_group)
print(f"MDE (80% power): {mde*100:.3f} pp")

# Sample size needed per group to detect our +0.5 pp threshold
threshold = 0.005
n_needed = ((z_alpha + z_beta)**2 * 2 * p_base * (1 - p_base)) / threshold**2
print(f"Users needed per group for +0.5 pp: {n_needed:,.0f}  (we had {n_per_group:,.0f})")

In [0]:
# --- Step 6: Daily stability ---
df["exposure_date"] = df["exposure_ts"].dt.date
daily = (df.groupby(["exposure_date", "test_group"])["converted"]
           .mean().unstack() * 100)
daily["diff_pp"] = daily["treatment"] - daily["control"]

print(daily.round(3))
print(f"\nDays treatment ahead: {(daily['diff_pp'] > 0).sum()} of {len(daily)}")